# MLP feedforward bagging ensemble

En esta notebook se hace bagging! yupi!  
Es la notebook que usamos en el concurso.

In [1]:
from pathlib import Path
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

from scripts import style
style.mpl_apply()

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from dataclasses import dataclass
import copy
import random
import math

import numpy as np
import pandas as pd


from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

from torch.utils.data import TensorDataset, DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

# Load the data

In [90]:
DATA_PATH = Path("09 features/redes_local/datos/processed/df.csv")

df = pd.read_csv(DATA_PATH)
df.head()

,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,max_R,...,pct_bright_block_2_2,H_shannon,center_mean_ratio,center_bright_ratio,lum_centroid_x,lum_centroid_y,filename,painter,genre,label
0,102.20103,105.277170,106.873116,32.077003,33.305286,33.510914,0.0,0.0,0.0,186.0,...,0.195138,-1419.565426,1.198618,1.749636,0.486834,-0.154286,Alfred_Sisley_1.jpg,Alfred_Sisley,Impresionismo,4
1,123.62030,136.959850,149.195050,46.771084,43.979904,49.263435,0.0,0.0,0.0,255.0,...,0.098290,-1419.565426,1.061136,0.937835,0.502996,-0.188640,Alfred_Sisley_10.jpg,Alfred_Sisley,Impresionismo,4
2,118.90262,116.895996,108.091415,48.649280,52.237015,61.140570,0.0,0.0,0.0,255.0,...,0.117654,-1419.489874,0.852622,1.311007,0.504956,-0.304950,Alfred_Sisley_11.jpg,Alfred_Sisley,Impresionismo,4
3,133.78065,140.773590,146.314380,36.072636,36.483162,50.136950,4.0,4.0,0.0,255.0,...,0.018207,-1419.565426,1.091522,1.702214,0.500399,-0.215292,Alfred_Sisley_12.jpg,Alfred_Sisley,Impresionismo,4
4,107.30281,108.520706,92.723560,32.748276,32.094944,37.935650,0.0,0.0,0.0,229.0,...,0.020639,-1419.565426,1.132043,2.128325,0.506062,-0.137083,Alfred_Sisley_13.jpg,Alfred_Sisley,Impresionismo,4


In [91]:
DATA_PATH = Path("09 features/redes_local/datos/processed/df2.csv")

df2 = pd.read_csv(DATA_PATH)
df2.head()

,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,max_R,...,lbp_hist_15,lbp_entropy,brightness_skew,ratio_dark,ratio_bright,ratio_warm,filename,painter,genre,label
0,102.20103,105.277170,106.873116,32.077003,33.305286,33.510914,0.0,0.0,0.0,186.0,...,0.011298,0.327957,-0.860794,0.183254,0.000189,0.240965,Alfred_Sisley_1.jpg,Alfred_Sisley,Impresionismo,4
1,123.62030,136.959850,149.195050,46.771084,43.979904,49.263435,0.0,0.0,0.0,255.0,...,0.011829,0.328205,-1.194858,0.081343,0.472129,0.195329,Alfred_Sisley_10.jpg,Alfred_Sisley,Impresionismo,4
2,118.90262,116.895996,108.091415,48.649280,52.237015,61.140570,0.0,0.0,0.0,255.0,...,0.011829,0.329164,-0.029142,0.244769,0.244209,0.523907,Alfred_Sisley_11.jpg,Alfred_Sisley,Impresionismo,4
3,133.78065,140.773590,146.314380,36.072636,36.483162,50.136950,4.0,4.0,0.0,255.0,...,0.011868,0.327928,-0.673457,0.054663,0.400458,0.176303,Alfred_Sisley_12.jpg,Alfred_Sisley,Impresionismo,4
4,107.30281,108.520706,92.723560,32.748276,32.094944,37.935650,0.0,0.0,0.0,229.0,...,0.011623,0.330782,-0.746116,0.116366,0.004845,0.457631,Alfred_Sisley_13.jpg,Alfred_Sisley,Impresionismo,4


In [92]:
columns = df.columns.tolist()
df = df2[columns]
df.head()

,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,max_R,...,pct_bright_block_2_2,H_shannon,center_mean_ratio,center_bright_ratio,lum_centroid_x,lum_centroid_y,filename,painter,genre,label
0,102.20103,105.277170,106.873116,32.077003,33.305286,33.510914,0.0,0.0,0.0,186.0,...,0.195138,-1419.565426,1.198618,1.749636,0.486834,-0.154286,Alfred_Sisley_1.jpg,Alfred_Sisley,Impresionismo,4
1,123.62030,136.959850,149.195050,46.771084,43.979904,49.263435,0.0,0.0,0.0,255.0,...,0.098290,-1419.565426,1.061136,0.937835,0.502996,-0.188640,Alfred_Sisley_10.jpg,Alfred_Sisley,Impresionismo,4
2,118.90262,116.895996,108.091415,48.649280,52.237015,61.140570,0.0,0.0,0.0,255.0,...,0.117654,-1419.489874,0.852622,1.311007,0.504956,-0.304950,Alfred_Sisley_11.jpg,Alfred_Sisley,Impresionismo,4
3,133.78065,140.773590,146.314380,36.072636,36.483162,50.136950,4.0,4.0,0.0,255.0,...,0.018207,-1419.565426,1.091522,1.702214,0.500399,-0.215292,Alfred_Sisley_12.jpg,Alfred_Sisley,Impresionismo,4
4,107.30281,108.520706,92.723560,32.748276,32.094944,37.935650,0.0,0.0,0.0,229.0,...,0.020639,-1419.565426,1.132043,2.128325,0.506062,-0.137083,Alfred_Sisley_13.jpg,Alfred_Sisley,Impresionismo,4


In [93]:
# Encode painters
painter_names = sorted(df["painter"].unique())
painter_to_idx = {name: i for i, name in enumerate(painter_names)}
idx_to_painter = {i: name for name, i in painter_to_idx.items()}

df["painter_idx"] = df["painter"].map(painter_to_idx)

NUM_PAINTERS = len(painter_names)

# Labels (movement / genre index)
# Assume df["label"] is already integer-coded per genre
NUM_LABELS = df["label"].nunique()

# Derive genre names per label (sorted by label index)
label_to_genre = (
    df.drop_duplicates("label")
      .sort_values("label")
      .set_index("label")["genre"]
      .to_dict()
)
class_names = [label_to_genre[i] for i in range(NUM_LABELS)]

print("Painters:", painter_names)
print("NUM_PAINTERS:", NUM_PAINTERS)
print("NUM_LABELS:", NUM_LABELS)
print("Labels → genres:", label_to_genre)


Painters: ['Alfred_Sisley', 'Camille_Pissarro', 'Caravaggio', 'Claude_Monet', 'Jackson_Pollock', 'Joan_Miro', 'Pablo_Picasso', 'Rembrandt', 'Rene_Magritte', 'Salvador_Dali', 'Vincent_van_Gogh']
NUM_PAINTERS: 11
NUM_LABELS: 5
Labels → genres: {0: 'Surrealismo', 1: 'Cubismo', 2: 'Expresionismo abstracto', 3: 'Barroco', 4: 'Impresionismo'}


C:\Users\herie\AppData\Local\Temp\ipykernel_1684\2115931354.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["painter_idx"] = df["painter"].map(painter_to_idx)


In [94]:
meta_cols = ["filename", "painter", "genre", "label", "painter_idx"]

feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].to_numpy(dtype=np.float32)
y_painter = df["painter_idx"].to_numpy(dtype=np.int64)
y_label   = df["label"].to_numpy(dtype=np.int64)

print("X shape:", X.shape)
print("y_painter shape:", y_painter.shape)
print("y_label shape:", y_label.shape)
print("Painter counts:", np.bincount(y_painter))
print("Label counts:",   np.bincount(y_label))

X shape: (200, 53)
y_painter shape: (200,)
y_label shape: (200,)
Painter counts: [20 10 10 30 15 20 30 20 10 20 15]
Label counts: [50 30 15 30 75]


# Train Set and Validation Set

In [95]:
X_train, X_val, y_p_train, y_p_val = train_test_split(
    X,
    y_painter,
    test_size=0.1,           # e.g. 80% train / 20% val
    stratify=y_painter,      # stratify by painter (since painter is first task)
)

print("Train X:", X_train.shape)
print("Val   X:", X_val.shape)

Train X: (180, 53)
Val   X: (20, 53)


## Scaling

In [96]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [97]:
BATCH_SIZE = 32

X_train_t = torch.from_numpy(X_train_scaled).float()
y_p_train_t = torch.from_numpy(y_p_train).long()

X_val_t = torch.from_numpy(X_val_scaled).float()
y_p_val_t = torch.from_numpy(y_p_val).long()

train_ds = TensorDataset(X_train_t, y_p_train_t)
val_ds   = TensorDataset(X_val_t,   y_p_val_t)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader)

(6, 1)

## Weights

In [99]:
painter_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_PAINTERS),
    y=y_p_train,
)

print("Painter class weights:", painter_class_weights)

painter_class_weights_t = torch.tensor(painter_class_weights, dtype=torch.float32, device=device)

Painter class weights: [0.90909091 1.81818182 1.81818182 0.60606061 1.25874126 0.90909091
 0.60606061 0.90909091 1.81818182 0.90909091 1.16883117]


# Architectures

## MLP

In [100]:
class PainterMLP_Slim(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_painters: int,
        bottleneck: int = 16,
        hidden_first: int = 32,
        dropout_p: float = 0.5,
    ):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_first),
            nn.BatchNorm1d(hidden_first),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_first, bottleneck),
            nn.Tanh(),
            nn.Dropout(dropout_p),
        )
        self.head = nn.Linear(bottleneck, num_painters)

    def forward(self, x):
        h = self.trunk(x)
        return self.head(h)


def make_model(hp, input_dim, num_painters, device):
    return PainterMLP_Slim(
        input_dim=input_dim,
        num_painters=num_painters,
        bottleneck=hp["bottleneck"],
        hidden_first=hp["hidden_first"],
        dropout_p=hp["dropout_p"],
    ).to(device)

# Training

## Criterion

In [101]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class WeightedCrossEntropy(nn.Module):
    def __init__(self, class_weights: torch.Tensor):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, logits, targets):
        return self.ce(logits, targets)


class WeightedLabelSmoothingCE(nn.Module):
    def __init__(self, class_weights: torch.Tensor, smoothing: float = 0.1):
        super().__init__()
        self.class_weights = class_weights
        self.smoothing = smoothing

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        C = logits.size(1)

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (C - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        w = self.class_weights[targets].unsqueeze(1)  # (B,1)
        loss = -(w * true_dist * log_probs).sum(dim=1).mean()
        return loss


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1.0 - pt) ** self.gamma) * ce
        return focal.mean()


def make_criterion(name: str, class_weights: torch.Tensor, smoothing=0.1, gamma=2.0):
    if name == "wce":
        return WeightedCrossEntropy(class_weights)
    if name == "wce_ls":
        return WeightedLabelSmoothingCE(class_weights, smoothing=smoothing)
    if name == "wfocal":
        return WeightedFocalLoss(class_weights, gamma=gamma)
    raise ValueError(f"Unknown criterion: {name}")

## Train

In [102]:

def train_one_epoch_painter(model, loader, optimizer, criterion, device):
    model.train()
    total, sum_loss = 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * xb.size(0)
        total += xb.size(0)
    return sum_loss / total


@torch.no_grad()
def eval_fold_painter(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def fit_and_score_fold_painter(
    X_tr, y_tr,
    X_va, y_va,
    hp,
    input_dim,
    num_painters,
    device,
    max_epochs=40,
    patience=8,
    batch_size=32,
    verbose=False,
):
    # scale per fold
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    # tensors
    X_tr_t = torch.from_numpy(X_tr_s).float()
    y_tr_t = torch.from_numpy(y_tr).long()
    X_va_t = torch.from_numpy(X_va_s).float()
    y_va_t = torch.from_numpy(y_va).long()

    # loaders
    tr_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                           batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va_t, y_va_t),
                           batch_size=batch_size, shuffle=False)

    # class weights on train fold
    w = compute_class_weight("balanced", classes=np.arange(num_painters), y=y_tr)
    w_t = torch.tensor(w, dtype=torch.float32, device=device)

    criterion = make_criterion(
        hp["criterion"],
        w_t,
        smoothing=hp.get("smoothing", 0.1),
        gamma=hp.get("gamma", 2.0),
    )

    model = make_model(hp, input_dim, num_painters, device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )

    best_state = None
    best_f1 = -np.inf
    best_acc = 0.0
    no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch_painter(model, tr_loader, optimizer, criterion, device)
        y_t, y_hat = eval_fold_painter(model, va_loader, device)

        val_f1  = f1_score(y_t, y_hat, average="macro")
        val_acc = accuracy_score(y_t, y_hat)

        if verbose:
            print(
                f"    epoch {epoch:02d} | train_loss={train_loss:.4f} | "
                f"val_f1_macro={val_f1:.3f} | val_acc={val_acc:.3f}"
            )

        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_f1, best_acc


## Best hyperparameters

In [103]:
# best_h={'bottleneck': 24, 'hidden_first': 32, 'dropout_p': 0.4, 'lr': 0.0024555426586822136, 'weight_decay': 4.070640977962409e-05, 'criterion': np.str_('wce_ls'), 'smoothing': 0.1}
best_h={'bottleneck': 32, 'hidden_first': 128, 'dropout_p': 0.2, 'lr': 0.006, 'weight_decay': 0.0009, 'criterion': np.str_('wce')}
best_h2 ={'bottleneck': 24, 'hidden_first': 48, 'dropout_p': 0.3, 'lr': 0.009809571718290793, 'weight_decay': 0.00017286437161330762, 'criterion': np.str_('wce')}
best_h3 = {'bottleneck': 24, 'hidden_first': 128, 'dropout_p': 0.1, 'lr': 0.002396152506176631, 'weight_decay': 0.000580016413870336, 'criterion': np.str_('wce')}
best_h4 = {'bottleneck': 64, 'hidden_first': 256, 'dropout_p': 0.2, 'lr': 0.0016339658583966275, 'weight_decay': 0.0006932564715531228, 'criterion': np.str_('wce')}


## Training

In [104]:
input_dim = X.shape[1]

def make_model(hp):
    return PainterMLP_Slim(
        input_dim=input_dim,
        num_painters=NUM_PAINTERS,
        bottleneck=hp["bottleneck"],
        hidden_first=hp["hidden_first"],
        dropout_p=hp["dropout_p"],
    ).to(device)

In [105]:
LR = best_h["lr"]
WEIGHT_DECAY = best_h["weight_decay"]

painter_criterion = make_criterion(
    name=best_h["criterion"],
    class_weights=painter_class_weights_t,
    smoothing=best_h.get("smoothing", 0.1),
)

In [106]:
K = 4
MAX_EPOCHS = 1000
PATIENCE = 250

In [107]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    painter_criterion,
    device,
):
    model.train()
    total = 0
    sum_loss = 0.0
    correct_painter = 0

    for X_batch, y_p_batch in loader:  
        X_batch = X_batch.to(device)
        y_p_batch = y_p_batch.to(device)

        optimizer.zero_grad()
        painter_logits = model(X_batch) 

        painter_loss = painter_criterion(painter_logits, y_p_batch)

        painter_loss.backward()
        optimizer.step()

        batch_size = X_batch.size(0)
        sum_loss += painter_loss.item() * batch_size
        total += batch_size

        painter_pred = painter_logits.argmax(dim=1)
        correct_painter += (painter_pred == y_p_batch).sum().item()

    epoch_loss = sum_loss / total
    painter_acc = correct_painter / total
    return epoch_loss, painter_acc


@torch.no_grad()
def evaluate(
    model,
    loader,
    painter_criterion,
    device,
):
    model.eval()
    total = 0
    sum_loss = 0.0
    correct_painter = 0

    for X_batch, y_p_batch in loader:
        X_batch = X_batch.to(device)
        y_p_batch = y_p_batch.to(device)

        painter_logits = model(X_batch)

        painter_loss = painter_criterion(painter_logits, y_p_batch)

        batch_size = X_batch.size(0)
        sum_loss += painter_loss.item() * batch_size
        total += batch_size

        painter_pred = painter_logits.argmax(dim=1)
        correct_painter += (painter_pred == y_p_batch).sum().item()

    epoch_loss = sum_loss / total
    painter_acc = correct_painter / total
    return epoch_loss, painter_acc


In [108]:
import numpy as np
import random
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score


def train_one_epoch_painter(model, loader, optimizer, criterion, device):
    model.train()
    total, sum_loss = 0, 0.0
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * xb.size(0)
        total += xb.size(0)

        y_true_all.append(yb.detach().cpu().numpy())
        y_pred_all.append(logits.argmax(1).detach().cpu().numpy())

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    epoch_loss = sum_loss / total
    epoch_acc  = accuracy_score(y_true_all, y_pred_all)
    epoch_f1   = f1_score(y_true_all, y_pred_all, average="macro")

    return epoch_loss, epoch_acc, epoch_f1


@torch.no_grad()
def evaluate_painter(model, loader, criterion, device):
    model.eval()
    total, sum_loss = 0, 0.0
    y_true_all, y_pred_all = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)

        sum_loss += loss.item() * xb.size(0)
        total += xb.size(0)

        y_true_all.append(yb.cpu().numpy())
        y_pred_all.append(logits.argmax(1).cpu().numpy())

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    epoch_loss = sum_loss / total
    epoch_acc  = accuracy_score(y_true_all, y_pred_all)
    epoch_f1   = f1_score(y_true_all, y_pred_all, average="macro")

    return epoch_loss, epoch_acc, epoch_f1

In [109]:
import numpy as np
import random
import copy

def train_single_model_with_history(
    max_epochs: int = 40,
    patience: int = 8,
    best_h: dict = None,
):
    # 1) model from best hyperparams
    model = make_model(best_h).to(device)

    # 2) optimizer from best hyperparams
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(best_h["lr"]),
        weight_decay=float(best_h["weight_decay"]),
    )

    try:
        # prefer global tensor if exists
        painter_weights_t = painter_class_weights_t
    except NameError:
        # otherwise compute from y_p_train
        w = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(NUM_PAINTERS),
            y=y_p_train,
        )
        painter_weights_t = torch.tensor(w, dtype=torch.float32, device=device)

    painter_criterion = make_criterion(
        best_h["criterion"],
        painter_weights_t,
        smoothing=float(best_h.get("smoothing", 0.1)),
        gamma=float(best_h.get("gamma", 2.0)),
    )

    history = {
        "train_loss": [],
        "train_painter_acc": [],
        "train_painter_f1": [],
        "val_loss": [],
        "val_painter_acc": [],
        "val_painter_f1": [],
    }

    best_state = None
    best_val_f1 = -np.inf
    epochs_no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_loss, train_acc, train_f1 = train_one_epoch_painter(
            model,
            train_loader,
            optimizer,
            painter_criterion,
            device,
        )

        val_loss, val_acc, val_f1 = evaluate_painter(
            model,
            val_loader,
            painter_criterion,
            device,
        )

        history["train_loss"].append(train_loss)
        history["train_painter_acc"].append(train_acc)
        history["train_painter_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_painter_acc"].append(val_acc)
        history["val_painter_f1"].append(val_f1)

        print(
            f"epoch {epoch:02d} | "
            f"train_loss={train_loss:.4f}, train_acc={train_acc:.3f}, train_f1={train_f1:.3f} | "
            f"val_loss={val_loss:.4f}, val_acc={val_acc:.3f}, val_f1={val_f1:.3f}"
        )

        # early stop on macro-F1 (matches CV selection)
        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history, best_val_f1


In [112]:
models, histories, scores = [], [], []

import time

start_time = time.time()

for k in range(K):
    best_h = [best_h, best_h2, best_h3, best_h4][k % 4]  # cycle through best hyperparams
    print(f"\n=== Training model {k+1}/{K} ===")
    model_k, hist_k, score_k = train_single_model_with_history(
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE, 
        best_h=best_h
    )
    models.append(model_k)
    histories.append(hist_k)
    scores.append(score_k)

print("Val painter acc of each model:", scores)
print("Mean:", np.mean(scores), "Std:", np.std(scores))


end_time = time.time()
print(f"Total training time for {K} models: {end_time - start_time:.2f} seconds") 


=== Training model 1/4 ===
epoch 01 | train_loss=2.1203, train_acc=0.249, train_f1=0.231 | val_loss=1.8455, val_acc=0.373, val_f1=0.322
epoch 02 | train_loss=1.7169, train_acc=0.385, train_f1=0.359 | val_loss=1.7057, val_acc=0.413, val_f1=0.364
epoch 03 | train_loss=1.5711, train_acc=0.450, train_f1=0.435 | val_loss=1.6024, val_acc=0.440, val_f1=0.412
epoch 04 | train_loss=1.4636, train_acc=0.483, train_f1=0.472 | val_loss=1.6134, val_acc=0.427, val_f1=0.440
epoch 05 | train_loss=1.4207, train_acc=0.489, train_f1=0.489 | val_loss=1.6305, val_acc=0.467, val_f1=0.457
epoch 06 | train_loss=1.3687, train_acc=0.502, train_f1=0.504 | val_loss=1.5595, val_acc=0.453, val_f1=0.446
epoch 07 | train_loss=1.2975, train_acc=0.544, train_f1=0.543 | val_loss=1.5548, val_acc=0.520, val_f1=0.531
epoch 08 | train_loss=1.2131, train_acc=0.566, train_f1=0.565 | val_loss=1.5778, val_acc=0.467, val_f1=0.469
epoch 09 | train_loss=1.1982, train_acc=0.582, train_f1=0.585 | val_loss=1.5553, val_acc=0.467, val_

# Test

In [137]:
# painter -> genre (take the most frequent genre per painter to be safe)
painter_to_genre = (
    df.groupby("painter")["genre"]
      .agg(lambda s: s.mode().iloc[0])   # mode handles any accidental duplicates
      .to_dict()
)

# optional: genre -> label (again mode-safe)
genre_to_label = (
    df.groupby("genre")["label"]
      .agg(lambda s: s.mode().iloc[0])
      .to_dict()
)

print(painter_to_genre)
print(genre_to_label)


{'Alfred_Sisley': 'Impresionismo', 'Camille_Pissarro': 'Impresionismo', 'Caravaggio': 'Barroco', 'Claude_Monet': 'Impresionismo', 'Jackson_Pollock': 'Expresionismo abstracto', 'Joan_Miro': 'Surrealismo', 'Pablo_Picasso': 'Cubismo', 'Rembrandt': 'Barroco', 'Rene_Magritte': 'Surrealismo', 'Salvador_Dali': 'Surrealismo', 'Vincent_van_Gogh': 'Impresionismo'}
{'Barroco': 3, 'Cubismo': 1, 'Expresionismo abstracto': 2, 'Impresionismo': 4, 'Surrealismo': 0}


In [138]:
import torch
import numpy as np

@torch.no_grad()
def ensemble_predict(models, X_t, device, return_logits=False):
    # accept numpy input
    if isinstance(X_t, np.ndarray):
        X_t = torch.from_numpy(X_t).float()

    X_t = X_t.to(device)

    p_logits_sum = None

    for model in models:
        model.eval()
        p_logits = model(X_t)
        p_logits_sum = p_logits if p_logits_sum is None else p_logits_sum + p_logits

    # average
    p_logits_mean = p_logits_sum / len(models)
    y_p_pred = p_logits_mean.argmax(dim=1).cpu().numpy()

    if return_logits:
        return y_p_pred, p_logits_mean
    return y_p_pred


In [139]:
TEST_PATH = Path("ValidacionNuriaHeriberto.csv")
df_test = pd.read_csv(TEST_PATH)

df_test.head()

,Unnamed: 0,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,...,lbp_hist_12,lbp_hist_13,lbp_hist_14,lbp_hist_15,lbp_entropy,brightness_skew,ratio_dark,ratio_bright,ratio_warm,filename
0,0,87.605370,100.523544,79.24710,57.309223,54.407196,51.809223,0.0,0.0,0.0,...,0.003777,0.001897,0.005037,0.011907,0.328007,-0.050906,0.349504,0.088890,0.086587,Alfred_Sisley_76.jpg
1,1,140.804950,149.116000,120.58856,37.960330,35.248108,45.345490,0.0,7.0,0.0,...,0.004855,0.001364,0.004915,0.011341,0.327792,0.013082,0.009253,0.266085,0.228543,Alfred_Sisley_77.jpg
2,2,103.524536,109.428210,101.00410,40.880750,35.311330,36.137894,0.0,7.0,0.0,...,0.005213,0.001350,0.004089,0.010921,0.329166,0.275512,0.152426,0.050435,0.272172,Alfred_Sisley_78.jpg
3,3,176.282320,160.992570,133.72836,30.512379,35.392464,38.867725,4.0,0.0,0.0,...,0.005355,0.001241,0.004203,0.011548,0.328344,-2.068637,0.019391,0.639367,0.990753,Alfred_Sisley_79.jpg
4,4,112.130050,114.630910,87.18911,37.541970,36.992800,35.506300,16.0,20.0,0.0,...,0.005217,0.001336,0.004836,0.011381,0.327460,0.056274,0.161534,0.046976,0.348908,Alfred_Sisley_80.jpg


In [140]:
# drop from columns
columns2 = [c for c in columns if c not in ['painter', 'genre', 'label']]
columns2

['mean_R',
 'mean_G',
 'mean_B',
 'std_R',
 'std_G',
 'std_B',
 'min_R',
 'min_G',
 'min_B',
 'max_R',
 'max_G',
 'max_B',
 'mean_S',
 'std_S',
 'mean_V',
 'std_V',
 'rms',
 'sobel_mean',
 'sobel_std',
 'sobel_pct_strong',
 'laplacian_mean_abs',
 'laplacian_std_abs',
 'gabor_mean_abs_0',
 'gabor_std_abs_0',
 'gabor_mean_abs_45',
 'gabor_std_abs_45',
 'gabor_mean_abs_90',
 'gabor_std_abs_90',
 'gabor_mean_abs_135',
 'gabor_std_abs_135',
 'mean_block_0_0',
 'pct_bright_block_0_0',
 'mean_block_0_1',
 'pct_bright_block_0_1',
 'mean_block_0_2',
 'pct_bright_block_0_2',
 'mean_block_1_0',
 'pct_bright_block_1_0',
 'mean_block_1_1',
 'pct_bright_block_1_1',
 'mean_block_1_2',
 'pct_bright_block_1_2',
 'mean_block_2_0',
 'pct_bright_block_2_0',
 'mean_block_2_1',
 'pct_bright_block_2_1',
 'mean_block_2_2',
 'pct_bright_block_2_2',
 'H_shannon',
 'center_mean_ratio',
 'center_bright_ratio',
 'lum_centroid_x',
 'lum_centroid_y',
 'filename']

In [141]:
dftest2 = df_test[columns2]
dftest2.head()

,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,max_R,...,mean_block_2_1,pct_bright_block_2_1,mean_block_2_2,pct_bright_block_2_2,H_shannon,center_mean_ratio,center_bright_ratio,lum_centroid_x,lum_centroid_y,filename
0,87.605370,100.523544,79.24710,57.309223,54.407196,51.809223,0.0,0.0,0.0,251.0,...,0.001604,0.130792,0.001710,0.157077,-1419.565426,0.854783,0.793376,0.550410,-0.278566,Alfred_Sisley_76.jpg
1,140.804950,149.116000,120.58856,37.960330,35.248108,45.345490,0.0,7.0,0.0,255.0,...,0.002196,0.014113,0.001809,0.005899,-1419.565426,1.088266,1.719288,0.504760,-0.216751,Alfred_Sisley_77.jpg
2,103.524536,109.428210,101.00410,40.880750,35.311330,36.137894,0.0,7.0,0.0,245.0,...,0.001650,0.194079,0.001721,0.146027,-1419.565426,1.039339,1.798396,0.493551,-0.164407,Alfred_Sisley_78.jpg
3,176.282320,160.992570,133.72836,30.512379,35.392464,38.867725,4.0,0.0,0.0,255.0,...,0.002059,0.069780,0.001950,0.020763,-1419.565426,1.047153,1.090197,0.485969,-0.217393,Alfred_Sisley_79.jpg
4,112.130050,114.630910,87.18911,37.541970,36.992800,35.506300,16.0,20.0,0.0,255.0,...,0.001579,0.000335,0.001660,0.000091,-1419.565426,0.822856,0.350550,0.534931,-0.159135,Alfred_Sisley_80.jpg


In [142]:
X_test = df_test[feature_cols].to_numpy(dtype=np.float32)
X_test_scaled = scaler.transform(X_test) 

X_test_t = torch.from_numpy(X_test_scaled).float()


In [143]:
y_p_pred_test = ensemble_predict(models, X_test_t, device)  # painter-only models

# painter index -> painter name
painter_pred_names = [idx_to_painter[i] for i in y_p_pred_test]

df_test["painter_pred"] = painter_pred_names
df_test.head()

,Unnamed: 0,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,...,lbp_hist_13,lbp_hist_14,lbp_hist_15,lbp_entropy,brightness_skew,ratio_dark,ratio_bright,ratio_warm,filename,painter_pred
0,0,87.605370,100.523544,79.24710,57.309223,54.407196,51.809223,0.0,0.0,0.0,...,0.001897,0.005037,0.011907,0.328007,-0.050906,0.349504,0.088890,0.086587,Alfred_Sisley_76.jpg,Claude_Monet
1,1,140.804950,149.116000,120.58856,37.960330,35.248108,45.345490,0.0,7.0,0.0,...,0.001364,0.004915,0.011341,0.327792,0.013082,0.009253,0.266085,0.228543,Alfred_Sisley_77.jpg,Alfred_Sisley
2,2,103.524536,109.428210,101.00410,40.880750,35.311330,36.137894,0.0,7.0,0.0,...,0.001350,0.004089,0.010921,0.329166,0.275512,0.152426,0.050435,0.272172,Alfred_Sisley_78.jpg,Salvador_Dali
3,3,176.282320,160.992570,133.72836,30.512379,35.392464,38.867725,4.0,0.0,0.0,...,0.001241,0.004203,0.011548,0.328344,-2.068637,0.019391,0.639367,0.990753,Alfred_Sisley_79.jpg,Vincent_van_Gogh
4,4,112.130050,114.630910,87.18911,37.541970,36.992800,35.506300,16.0,20.0,0.0,...,0.001336,0.004836,0.011381,0.327460,0.056274,0.161534,0.046976,0.348908,Alfred_Sisley_80.jpg,Salvador_Dali


In [144]:
df_test["genre_pred"] = df_test["painter_pred"].map(painter_to_genre)
df_test["label_pred"] = df_test["genre_pred"].map(genre_to_label)  # optional

df_test[["painter_pred", "genre_pred", "label_pred"]].head()

,painter_pred,genre_pred,label_pred
0,Claude_Monet,Impresionismo,4
1,Alfred_Sisley,Impresionismo,4
2,Salvador_Dali,Surrealismo,0
3,Vincent_van_Gogh,Impresionismo,4
4,Salvador_Dali,Surrealismo,0


In [145]:
# save only the column label_pred
df_test[['label_pred']].to_csv("NuriaWaliHeri2.csv", index=False)